# Expert debugger: end-to-end pipeline
Failed run → evidence → issue detector → ranker → knowledge layer → (RCA agent → matcher) → confidence gate → human approval. This notebook wires the CPU parts together on synthetic data and marks where the two LLM stages plug in (see the 04/05 notebooks).

> All data in this notebook is **synthetic** (generated below). The real system runs on private validation data; the code paths are the same, the numbers in the README are from the real data.

In [ ]:
!pip -q install scikit-learn matplotlib numpy scipy

In [ ]:
%%writefile synthetic.py
"""Synthetic failure generator for the demo notebooks (no real data is published).
Each bug has a signature (2-3 log lines), a domain, an owner and a test family; runs of that bug emit the signature
mixed with background noise, sometimes miss a line, and 'sibling' bugs share most of their signature, which is the
hard case in the real system. Not-a-bug runs are infrastructure or test-content problems."""
import random
from datetime import datetime, timedelta

DOMAINS = ["power", "memory", "reset", "virtualization", "media", "compute", "fabric", "firmware"]
COMPONENTS = ["gt", "dma", "pcie", "fw", "mem", "sched", "link", "irq", "pmu", "vf"]
SYMPTOMS = ["engine reset", "timeout waiting for fence", "page fault at <HEX>", "device lost", "link training failed",
            "ECC uncorrectable", "watchdog expired", "CRC mismatch", "thermal trip", "context banned"]
NOISE = ["test step <N> started", "allocating <N> buffers", "INFO: iteration <N> passed", "loading module", "heartbeat ok",
         "perf counter sample <N>", "config applied", "mounting share", "INFO: retrying connection"]
GENERIC = ["FAIL: test step <N> returned non-zero", "ERROR: timeout waiting for test harness", "FAIL: assertion in test script"]
INFRA = ["ERROR: node unreachable", "ERROR: container image pull failed", "FAIL: missing test binary", "ERROR: disk quota exceeded"]


def make_world(n_bugs=40, n_runs=1200, p_none=.18, seed=0):
    rng = random.Random(seed)
    t0 = datetime(2026, 1, 1)
    bugs = []
    for i in range(n_bugs):
        sib = bugs[rng.randrange(len(bugs))] if bugs and rng.random() < .35 else None
        comp = sib["component"] if sib else rng.choice(COMPONENTS)
        sig = list(sib["signature"][:2]) if sib else [f"{comp}: {rng.choice(SYMPTOMS)}", f"{comp}: {rng.choice(SYMPTOMS)} on <PCI>"]
        sig.append(f"{comp}: first fault code {rng.randrange(16, 255):#x}")
        bugs.append({"id": f"BUG-{i:03d}", "domain": sib["domain"] if sib else rng.choice(DOMAINS), "component": comp,
                     "owner": f"owner{rng.randrange(8)}", "family": f"T{rng.randrange(12)}", "signature": sig,
                     "created": t0 + timedelta(days=rng.randrange(0, 200)),
                     "title": f"[{comp}] {sig[0].split(': ', 1)[1]}", "root_cause": f"{comp} {sig[-1].split(': ', 1)[1]}"})
    w = [1 / (i + 1) ** .8 for i in range(n_bugs)]          # long tail: a few bugs own most failures
    runs = []
    for j in range(n_runs):
        start = t0 + timedelta(days=rng.uniform(0, 260))
        if rng.random() < p_none:
            lines, label, b = ([rng.choice(INFRA)] if rng.random() < .6 else [rng.choice(GENERIC)]), "NONE", None
        else:
            b = rng.choices(bugs, w)[0]
            if b["created"] > start + timedelta(days=30):
                start = b["created"] + timedelta(days=rng.uniform(0, 30))
            lines, label = [s for s in b["signature"] if rng.random() > .3] or b["signature"][:1], b["id"]
            if rng.random() < .25:
                lines.append(rng.choice(GENERIC + INFRA))      # product bugs also trip generic/infra-looking lines
        lines += rng.sample(NOISE, 4)
        rng.shuffle(lines)
        runs.append({"run": f"RUN-{j:05d}", "label": label, "start": start, "lines": lines,
                     "domain": b["domain"] if b else rng.choice(DOMAINS), "owner": b["owner"] if b else f"owner{rng.randrange(8)}",
                     "family": b["family"] if b and rng.random() < .7 else f"T{rng.randrange(12)}"})
    return bugs, runs


def split(runs, bugs, seed=0):
    """Grouped by bug: 30% of bugs are test-only, so the test split measures bugs never seen in training."""
    rng = random.Random(seed)
    ids = [b["id"] for b in bugs]
    rng.shuffle(ids)
    test_bugs = set(ids[: int(.3 * len(ids))])
    for r in runs:
        r["split"] = "test" if (r["label"] in test_bugs or (r["label"] == "NONE" and rng.random() < .3)) else "train"
        # frequent bugs also appear in test (seen-bug slice)
        if r["label"] not in test_bugs and r["label"] != "NONE" and rng.random() < .2:
            r["split"] = "test"
    return runs

In [ ]:
%%writefile data_pipeline.py
"""Data-collection pipeline: failed CI runs -> normalised evidence -> leak-free labelled datasets.

Every model in the system trains and is evaluated on what this module produces. The sources are interfaces:
plug in your CI server, log store and bug tracker. Nothing here calls an LLM.

    runs  = ci.failed_runs(since)                       # failed test runs + metadata
    ev    = collect_evidence(run, logs)                 # windowed, normalised, deduplicated log lines
    rows  = build_dataset(runs, tracker, logs)          # one JSONL row per run, with its label and split
"""
from __future__ import annotations

import hashlib
import json
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Iterable, Protocol


# ---------------------------------------------------------------- sources (interfaces)
@dataclass
class Run:
    id: str
    test: str              # test case / suite id
    domain: str            # feature area (power, memory, reset, ...)
    host: str
    start: datetime
    end: datetime
    reason: str            # one-line failure reason from the harness
    meta: dict = field(default_factory=dict)   # owner, config, build, ...


@dataclass
class Bug:
    id: str
    title: str
    description: str
    created: datetime
    state: str


class CI(Protocol):
    def failed_runs(self, since: datetime) -> Iterable[Run]: ...
    def artifact_text(self, run: Run, name: str) -> str | None: ...


class LogStore(Protocol):
    def search(self, index: str, host: str, start: datetime, end: datetime, size: int = 2000) -> list[str]: ...


class Tracker(Protocol):
    def links(self) -> dict[str, tuple[str, datetime]]: ...   # run id -> (bug id, link time)
    def bug(self, bug_id: str) -> Bug: ...


# ---------------------------------------------------------------- normalisation
_MASKS = [
    (re.compile(r"^\[?\s*\d+\.\d+\]\s*"), ""),                                          # kernel monotonic stamp
    (re.compile(r"\d{4}-\d\d-\d\d[T ]\d\d:\d\d:\d\d(\.\d+)?(Z|[+-]\d\d:?\d\d)?"), "<TS>"),
    (re.compile(r"\b[0-9a-f]{4}:[0-9a-f]{2}:[0-9a-f]{2}\.[0-7]\b", re.I), "<PCI>"),
    (re.compile(r"\b[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}\b", re.I), "<UUID>"),
    (re.compile(r"\b0x[0-9a-f]+\b", re.I), "<HEX>"),
    (re.compile(r"\b[a-z]+-?\d{2,}[a-z0-9-]*\b", re.I), "<HOST>"),
    (re.compile(r"\b\d{3,}\b"), "<N>"),
]
_SIGNAL = re.compile(r"error|fail|fatal|timeout|hang|lost|reset|panic|oops|corrupt|denied|mismatch|assert", re.I)


def normalise(line: str) -> str:
    """Mask volatile tokens so the same failure gives the same string on every host and day."""
    s = line.strip()
    for rx, rep in _MASKS:
        s = rx.sub(rep, s)
    return re.sub(r"\s+", " ", s)


def skeleton(line: str) -> str:
    """Hash key of a normalised line (used for dedup, clustering and signature matching)."""
    return hashlib.sha1(normalise(line).lower().encode()).hexdigest()[:16]


# ---------------------------------------------------------------- evidence
INDEXES = ("kernel", "exec", "tracked", "alerts")     # kernel log, execution log, harness failure records, alerts
PAD = timedelta(minutes=15)


def collect_evidence(run: Run, logs: LogStore, ci: CI | None = None, cap: int = 40) -> dict[str, list[str]]:
    """Run-scoped evidence: every index in [start - 15 min, end + 15 min] on the run's host, normalised,
    deduplicated by skeleton, signal lines first, capped per source. Falls back to CI artifacts when an
    index has no data for the run (some suites never reach the log store)."""
    out = {}
    for ix in INDEXES:
        lines = logs.search(ix, run.host, run.start - PAD, run.end + PAD)
        if not lines and ci is not None:
            lines = (ci.artifact_text(run, f"{ix}.log") or "").splitlines()
        seen, keep = set(), []
        for ln in sorted(lines, key=lambda l: not _SIGNAL.search(l)):
            k = skeleton(ln)
            if k not in seen:
                seen.add(k)
                keep.append(normalise(ln))
            if len(keep) >= cap:
                break
        out[ix] = keep
    return out


def first_fault(ev: dict[str, list[str]]) -> str | None:
    """Earliest signal line across sources: a cheap, deterministic proxy of the root-cause line."""
    for ix in ("kernel", "exec", "tracked"):
        for ln in ev.get(ix, []):
            if _SIGNAL.search(ln):
                return ln
    return None


# ---------------------------------------------------------------- labels and leak-free splits
def build_dataset(runs: list[Run], tracker: Tracker, logs: LogStore, ci: CI | None = None, seed: int = 0,
                  big_group: int = 8, frac=(0.7, 0.1, 0.2)) -> list[dict]:
    """One row per failed run.
    label   : linked bug id, or NONE when the run was triaged as not-a-bug.
    split   : grouped by bug, so a bug is in exactly one of train / val / test (measures generalisation to
              bugs never seen in training). Very large groups are split by run instead, so frequent bugs are
              represented in every split.
    cutoff  : the run's start time. Any retrieval or tool built on top must hide bugs filed after it, and
              must hide the run's own link (prevents label leakage through the knowledge base)."""
    links = tracker.links()
    rows = []
    for r in runs:
        bug, linked = links.get(r.id, (None, None))
        rows.append({"run": r.id, "test": r.test, "domain": r.domain, "reason": normalise(r.reason), "meta": r.meta,
                     "evidence": collect_evidence(r, logs, ci), "label": bug or "NONE",
                     "cutoff": r.start.isoformat(), "linked": linked.isoformat() if linked else None})
    rng = random.Random(seed)
    groups = defaultdict(list)
    for row in rows:
        groups[row["label"]].append(row)
    keys = sorted(k for k in groups if k != "NONE")
    rng.shuffle(keys)
    n = len(keys)
    cut = [int(frac[0] * n), int((frac[0] + frac[1]) * n)]
    for i, k in enumerate(keys):
        g = groups[k]
        if len(g) >= big_group:
            for j, row in enumerate(sorted(g, key=lambda x: x["cutoff"])):
                row["split"] = "train" if j < .7 * len(g) else ("val" if j < .8 * len(g) else "test")
        else:
            for row in g:
                row["split"] = "train" if i < cut[0] else ("val" if i < cut[1] else "test")
    for row in groups.get("NONE", []):
        row["split"] = rng.choices(["train", "val", "test"], frac)[0]
    return rows


def unseen_mask(rows: list[dict]) -> list[bool]:
    """True for evaluation rows whose bug never appears in train: the honest 'new failure' slice."""
    train_bugs = {r["label"] for r in rows if r["split"] == "train"}
    return [r["split"] != "train" and r["label"] != "NONE" and r["label"] not in train_bugs for r in rows]


def out_of_fold(rows: list[dict], fit, predict, folds: int = 5, seed: int = 0) -> dict[str, list[str]]:
    """Out-of-fold candidate lists for training downstream models: the matcher must learn from the ranker's
    real mistakes, not from a ranker that has already seen the answer."""
    bugs = sorted({r["label"] for r in rows})
    random.Random(seed).shuffle(bugs)
    fold_of = {b: i % folds for i, b in enumerate(bugs)}
    out = {}
    for f in range(folds):
        model = fit([r for r in rows if fold_of[r["label"]] != f])
        for r in rows:
            if fold_of[r["label"]] == f:
                out[r["run"]] = predict(model, r)
    return out


def write_jsonl(rows: list[dict], path: str) -> None:
    with open(path, "w") as fh:
        for r in rows:
            fh.write(json.dumps(r, default=str) + "\n")
    print(path, len(rows), Counter(r["split"] for r in rows))

In [ ]:
%%writefile ranker.py
"""Failure ranker (no LLM): failed run -> ranked list of known bugs + calibrated LINK / REVIEW / NEW decision.

Features per (run, candidate bug) pair:
  - log channel   : signature lines of the bug found in the run's normalised evidence (exact skeleton hits,
                    fuzzy phrase hits, idf-weighted), first-fault agreement
  - metadata prior: same test family, same domain, same owner, peer votes (how many earlier runs of this test
                    family were linked to the bug), recency of the bug's last link, bug created before the run
  - hard rules    : a bug tagged for another platform vendor never outranks any other candidate
Model: logistic regression over pair features (pairwise -> pointwise scores), grouped CV by bug, then a
calibrated open-set threshold: LINK only when P(top-1) >= tau, tau picked for >= 85 % out-of-fold precision.
"""
from __future__ import annotations

import math
import re
from collections import Counter, defaultdict

import numpy as np
from sklearn.linear_model import LogisticRegression

FEATS = ["sig_hits", "sig_frac", "sig_idf", "first_fault", "same_family", "same_domain", "same_owner",
         "peer_votes", "log_recency", "created_before"]


def tokens(s: str) -> set[str]:
    return set(re.findall(r"[a-z_]{3,}", s.lower()))


class Ranker:
    def __init__(self, C: float = 1.0, pool: int = 50):
        self.C, self.pool = C, pool

    # ------------------------------------------------------------- fit
    def fit(self, runs: list[dict], bugs: list[dict]):
        self.bugs = {b["id"]: b for b in bugs}
        self.df = Counter(t for b in bugs for s in b["signature"] for t in tokens(s))
        self.history = sorted((r for r in runs if r["label"] != "NONE"), key=lambda r: r["start"])
        X, y = [], []
        for r in runs:
            if r["label"] == "NONE":
                continue
            for bid in self._pool(r):
                X.append(self._feat(r, self.bugs[bid]))
                y.append(int(bid == r["label"]))
        self.clf = LogisticRegression(C=self.C, max_iter=2000, class_weight="balanced").fit(np.array(X), np.array(y))
        return self

    # ------------------------------------------------------------- features
    def _pool(self, r):
        return [b for b, v in self.bugs.items() if v["created"] <= r["start"]][: self.pool * 10]

    def _feat(self, r, b):
        text = " ".join(r["lines"]).lower()
        hits = [s for s in b["signature"] if s.lower() in text]
        idf = sum(math.log(1 + len(self.bugs) / (1 + self.df[t])) for s in hits for t in tokens(s))
        prev = [h for h in self.history if h["start"] < r["start"] and h["label"] == b["id"]]
        peers = sum(1 for h in prev if h["family"] == r["family"])
        last = (r["start"] - prev[-1]["start"]).days if prev else 999
        ff = next((l for l in r["lines"] if re.search(r"fault|reset|lost|timeout|mismatch", l)), "")
        return [len(hits), len(hits) / len(b["signature"]), idf, int(bool(ff) and ff in b["signature"]),
                int(b["family"] == r["family"]), int(b["domain"] == r["domain"]), int(b["owner"] == r["owner"]),
                math.log1p(peers), -math.log1p(last), int(b["created"] <= r["start"])]

    # ------------------------------------------------------------- predict
    def rank(self, r, k: int = 5):
        pool = self._pool(r)
        if not pool:
            return []
        p = self.clf.predict_proba(np.array([self._feat(r, self.bugs[b]) for b in pool]))[:, 1]
        order = np.argsort(-p)[:k]
        return [(pool[i], float(p[i])) for i in order]


def calibrate_tau(scored: list[tuple[float, bool]], target_precision: float = .85) -> float:
    """Lowest threshold whose out-of-fold precision on LINK decisions is >= target."""
    for tau in np.arange(.05, 1.0, .01):
        sel = [ok for s, ok in scored if s >= tau]
        if sel and sum(sel) / len(sel) >= target_precision:
            return float(tau)
    return 1.0


def evaluate(ranker: Ranker, runs: list[dict], unseen: set[str] | None = None) -> dict:
    res = defaultdict(list)
    for r in runs:
        if r["label"] == "NONE":
            continue
        top = [b for b, _ in ranker.rank(r, 3)]
        res["top1"].append(bool(top) and top[0] == r["label"])
        res["top3"].append(r["label"] in top)
        if unseen is not None and r["label"] in unseen:
            res["unseen_top1"].append(bool(top) and top[0] == r["label"])
    return {k: round(100 * sum(v) / len(v), 1) for k, v in res.items() if v}

In [ ]:
%%writefile detector.py
"""Test-content / infrastructure issue detector: is this failure *not* a product bug?

A false positive hides a real bug, so the model only fires at an operating point with >= 90 % out-of-fold
precision. Features: TF-IDF over the failure reason, the failed stage and the error lines of the execution log
(numbers and paths masked), plus one-hot categorical flags (test family, owner, domain, evidence present or not).
Labels: runs humans tagged as test-content issues (positives) vs runs linked to a product bug (negatives).
"""
from __future__ import annotations

import numpy as np
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import OneHotEncoder

CATS = ["family", "owner", "domain", "has_logs"]


class IssueDetector:
    def __init__(self, C: float = 3.0, min_precision: float = .9):
        self.C, self.min_precision = C, min_precision

    def _text(self, rows):
        return [" ".join(r["lines"]) for r in rows]

    def _cats(self, rows):
        return [[str(r.get(c, "")) for c in CATS] for r in rows]

    def _x(self, rows, fit=False):
        if fit:
            self.tf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
            self.oh = OneHotEncoder(handle_unknown="ignore")
            return hstack([self.tf.fit_transform(self._text(rows)), self.oh.fit_transform(self._cats(rows))]).tocsr()
        return hstack([self.tf.transform(self._text(rows)), self.oh.transform(self._cats(rows))]).tocsr()

    def fit(self, rows, y, groups):
        """Grouped CV gives out-of-fold scores; the threshold is the best-recall point with OOF precision >= target."""
        oof = np.zeros(len(rows))
        for tr, te in GroupKFold(5).split(rows, y, groups):
            m = LogisticRegression(C=self.C, max_iter=3000, class_weight="balanced")
            m.fit(self._x([rows[i] for i in tr], fit=True), np.array(y)[tr])
            oof[te] = m.predict_proba(self._x([rows[i] for i in te]))[:, 1]
        p, r, t = precision_recall_curve(y, oof)
        ok = [(r[i], t[i]) for i in range(len(t)) if p[i] >= self.min_precision]
        self.tau = max(ok)[1] if ok else 1.0
        self.oof_ap = average_precision_score(y, oof)
        self.oof_recall = max(ok)[0] if ok else 0.0
        self.model = LogisticRegression(C=self.C, max_iter=3000, class_weight="balanced").fit(self._x(rows, fit=True), y)
        return self

    def flag(self, row) -> tuple[bool, float]:
        s = float(self.model.predict_proba(self._x([row]))[:, 1][0])
        return s >= self.tau, s

In [ ]:
%%writefile knowledge.py
"""Knowledge layer: structured bug cards + hybrid search + a small knowledge graph.

1. Bug cards (LLM, once per bug, cached): free-text bug reports are long and noisy (console pastes, HTML).
   An LLM turns each into a card: component, symptom, first fault, match phrases (verbatim log lines),
   distinguishing detail vs sibling bugs, workaround. Cards are validated: a match phrase that never occurs in
   any linked run's evidence is dropped.
2. Hybrid search: BM25 (exact log vocabulary) + dense embeddings (paraphrase) fused with reciprocal-rank fusion.
   Used by the agent as a tool and to widen the ranker's candidate pool.
3. Knowledge graph: nodes = bugs, components, tests, owners, root causes; edges = links, supersedes, shares
   component, co-occurs. Queried by the agent for neighbourhoods ("siblings of this bug").
Everything respects the run's cutoff: bugs filed after the run, and the run's own link, are invisible.
"""
from __future__ import annotations

import json
import math
import re
from collections import Counter, defaultdict

CARD_PROMPT = """You turn one bug report into a structured card for a log-matching system.
Return JSON with keys: component, symptom, first_fault, match_phrases (3-8 verbatim log lines that identify THIS bug),
distinguishing (what separates it from similar bugs), workaround.
Only copy phrases that appear verbatim in the report. Do not invent log lines.

Report:
{report}"""


def make_card(report: str, llm) -> dict:
    card = json.loads(llm(CARD_PROMPT.format(report=report[:12000])))
    card["match_phrases"] = [p for p in card.get("match_phrases", []) if p and p in report]  # verbatim only
    return card


def validate_card(card: dict, linked_evidence: list[str]) -> dict:
    """Drop phrases that never hit a run linked to this bug (they would only add noise to retrieval)."""
    text = "\n".join(linked_evidence).lower()
    card["match_phrases"] = [p for p in card["match_phrases"] if p.lower() in text] or card["match_phrases"][:2]
    return card


# ------------------------------------------------------------------ hybrid search
def _tok(s):
    return re.findall(r"[a-z0-9_]{2,}", s.lower())


class BM25:
    def __init__(self, docs: list[str], k1=1.4, b=.75):
        self.d = [_tok(x) for x in docs]
        self.avg = sum(map(len, self.d)) / max(1, len(self.d))
        self.df = Counter(t for doc in self.d for t in set(doc))
        self.k1, self.b, self.N = k1, b, len(docs)

    def scores(self, q: str):
        qt = _tok(q)
        out = []
        for doc in self.d:
            tf = Counter(doc)
            s = 0.0
            for t in qt:
                if t in tf:
                    idf = math.log(1 + (self.N - self.df[t] + .5) / (self.df[t] + .5))
                    s += idf * tf[t] * (self.k1 + 1) / (tf[t] + self.k1 * (1 - self.b + self.b * len(doc) / self.avg))
            out.append(s)
        return out


def rrf(*rankings: list[int], k: int = 60) -> list[int]:
    """Reciprocal-rank fusion of several ranked id lists."""
    score = defaultdict(float)
    for r in rankings:
        for pos, i in enumerate(r):
            score[i] += 1 / (k + pos + 1)
    return sorted(score, key=lambda i: -score[i])


class KB:
    def __init__(self, cards: list[dict], embed=None):
        self.cards = cards
        self.docs = [" ".join([c["title"], c.get("symptom", ""), c.get("first_fault", ""), *c.get("match_phrases", [])]) for c in cards]
        self.bm25 = BM25(self.docs)
        self.embed = embed
        self.E = embed(self.docs) if embed else None

    def search(self, query: str, k: int = 10, cutoff=None, hide: set | None = None) -> list[dict]:
        s = self.bm25.scores(query)
        bm = sorted(range(len(self.docs)), key=lambda i: -s[i])
        lists = [bm]
        if self.embed is not None:
            q = self.embed([query])[0]
            sims = (self.E @ q) / ((self.E ** 2).sum(1) ** .5 * (q @ q) ** .5 + 1e-9)
            lists.append(list(sims.argsort()[::-1]))
        out = []
        for i in rrf(*lists):
            c = self.cards[i]
            if (cutoff and c["created"] > cutoff) or (hide and c["id"] in hide):
                continue                                   # time cut + label hiding: no leakage
            out.append(c)
            if len(out) == k:
                break
        return out


# ------------------------------------------------------------------ knowledge graph
class Graph:
    def __init__(self):
        self.adj = defaultdict(set)

    def add(self, a, rel, b):
        self.adj[a].add((rel, b))
        self.adj[b].add((f"~{rel}", a))

    def neighbourhood(self, node, depth=2):
        seen, frontier = {node}, [node]
        for _ in range(depth):
            nxt = []
            for n in frontier:
                for _, m in self.adj[n]:
                    if m not in seen:
                        seen.add(m)
                        nxt.append(m)
            frontier = nxt
        return seen - {node}

    def siblings(self, bug):
        comps = {b for r, b in self.adj[bug] if r == "component"}
        return {x for c in comps for r, x in self.adj[c] if r == "~component" and x != bug}

In [ ]:
import random
from collections import Counter
from synthetic import make_world, split
from ranker import Ranker, calibrate_tau
from detector import IssueDetector
from knowledge import KB
bugs, runs = make_world(n_bugs=50, n_runs=1800, p_none=.2, seed=7); runs = split(runs, bugs)
for r in runs: r["has_logs"] = True
train = [r for r in runs if r["split"] == "train"]; test = [r for r in runs if r["split"] == "test"]
det = IssueDetector().fit(train, [int(r["label"] == "NONE") for r in train], [r["label"] if r["label"] != "NONE" else r["family"] for r in train])
rk = Ranker().fit(train, bugs)
kb = KB([{**b, "symptom": b["signature"][0], "first_fault": b["signature"][-1], "match_phrases": b["signature"]} for b in bugs])
tau = calibrate_tau([(rk.rank(r, 1)[0][1], rk.rank(r, 1)[0][0] == r["label"]) for r in train if r["label"] != "NONE" and rk.rank(r, 1)], .85)

In [ ]:
def decide(r):
    flag, _ = det.flag(r)
    if flag: return "NOT-A-BUG", None
    top = rk.rank(r, 5)
    pool = {b for b, _ in top} | {c["id"] for c in kb.search(" ".join(r["lines"]), 5, cutoff=r["start"])}
    if top and top[0][1] >= tau: return "LINK", top[0][0]
    # here the production system runs the RCA agent + matcher on `pool`, behind the confidence gate
    return "REVIEW", top[0][0] if top else None
dec = [(r, *decide(r)) for r in test]
print(Counter(d for _, d, _ in dec))
link = [(r, b) for r, d, b in dec if d == "LINK"]
print(f"auto-LINK precision {sum(r['label'] == b for r, b in link) / max(1, len(link)):.1%} on {len(link)} runs")
nab = [r for r, d, _ in dec if d == "NOT-A-BUG"]
print(f"NOT-A-BUG precision {sum(r['label'] == 'NONE' for r in nab) / max(1, len(nab)):.1%} on {len(nab)} runs")